In [1]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from statsmodels.tsa.stattools import coint, adfuller
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
np.random.seed(42)

print('All imports successful.')

# Strategy Parameters
ZSCORE_WINDOW  = 52
EXIT_THRESHOLD = 0.5
STOP_LOSS      = 3.5

tier1_pairs = [
    ('CDNS', 'SNPS'),
    ('KLAC', 'ODFL'),
    ('KLAC', 'SNPS'),
    ('ODFL', 'SNPS'),
    ('CDNS', 'KLAC'),
]

tier2_pairs = [
    ('FAST', 'MSFT'),
    ('QCOM', 'TSLA'),
    ('PAYX', 'PEP'),
    ('ADP', 'PEP'),
    ('AMGN', 'CTAS'),
    ('CDW', 'CTAS'),
    ('AMGN', 'CDW'),
]

all_pairs = tier1_pairs + tier2_pairs

All imports successful.


In [2]:
# Re-downloading full price history
import yfinance as yf

tickers = list(set([t for pair in all_pairs for t in pair]))
tickers.sort()

print(f'Downloading price data for {len(tickers)} unique tickers...')
print(f'Tickers: {','.join(tickers)}\n')

raw = yf.download(
    tickers,
    start='2014-01-01',
    end='2024-01-01',
    interval='1wk',
    auto_adjust=True,
    progress=True
)

price_data = raw['Close']
price_data = price_data.resample('W-WED').last()
price_data = price_data.dropna(how='all')
price_data = price_data.ffill().dropna(axis=1)

print(f'\nPrice data shape: {price_data.shape}')
print(f'Date range: {price_data.index[0].date()} to {price_data.index[-1].date()}')
print(f'Tickers retained: {len(price_data.columns)}')
print(f'\nAll tickers present: {all(t in price_data.columns for t in tickers)}')

Tickers: ADP,AMGN,CDNS,CDW,CTAS,FAST,KLAC,MSFT,ODFL,PAYX,PEP,QCOM,SNPS,TSLA



[*********************100%***********************]  14 of 14 completed


Price data shape: (522, 14)
Date range: 2014-01-01 to 2023-12-27
Tickers retained: 14

All tickers present: True


In [3]:
# Defining the three periods: Train, Validate, Test
TRAIN_START = '2014-01-01'
TRAIN_END = '2017-12-31'
VAL_START = '2018-01-01'
VAL_END = '2020-12-31'
TEST_START = '2021-01-01'
TEST_END = '2023-12-31'

# Spliting price data into three periods
train_prices = price_data[TRAIN_START:TRAIN_END]
val_prices = price_data[VAL_START:VAL_END]
test_prices = price_data[TEST_START:TEST_END]

# Printing period summaries
print('Three-Period Data Split')
print('='*70)

for name, df, start, end in [
    ('Training', train_prices, TRAIN_START, TRAIN_END),
    ('Validating', val_prices, VAL_START, VAL_END),
    ('Testing', test_prices, TEST_START, TEST_END)
]:
    print(f'\n{name} Period ({start} to {end}):')
    print(f'  Weeks: {len(df)}')
    print(f'  Start: {df.index[0].date()}')
    print(f'  End: {df.index[-1].date()}')
    print(f'  Tickers: {len(df.columns)}')

Three-Period Data Split

Training Period (2014-01-01 to 2017-12-31):
  Weeks: 209
  Start: 2014-01-01
  End: 2017-12-27
  Tickers: 14

Validating Period (2018-01-01 to 2020-12-31):
  Weeks: 157
  Start: 2018-01-03
  End: 2020-12-30
  Tickers: 14

Testing Period (2021-01-01 to 2023-12-31):
  Weeks: 156
  Start: 2021-01-06
  End: 2023-12-27
  Tickers: 14


In [4]:
def estimate_hedge_ratio(ticker1, ticker2, train_prices):
    """
    Estimates the OLS hedge ratio between two stocks using
    only the training period price data.

    This is the most critical methodological constraint in Notebook 4.
    The hedge ratio is frozen after training and applied
    unchanged to the validation and test periods.

    Returns hedge ratio, intercept and R-squared
    """
    combined = pd.concat(
        [train_prices[ticker1], train_prices[ticker2]], axis=1
    ).dropna()
    combined.columns = [ticker1, ticker2]

    X     = add_constant(combined[ticker2])
    model = OLS(combined[ticker1], X).fit()

    hedge_ratio = float(model.params[ticker2])
    intercept   = float(model.params['const'])
    r_squared   = float(model.rsquared)

    return hedge_ratio, intercept, r_squared

# Compute and store hedge ratios for all 12 pairs
print('Estimating hedge ratios on training data only (2014-2017)...')
print('These ratios will be frozen and applied to all three periods. \n')

hedge_ratios = {}

print(f'{'Pair': <15} {'Hedge Ratio':>12} {'Intercept':>12} {'R-Squared':>12}')
print('-'*70)

for t1, t2 in all_pairs:
    pair_name = f'{t1}/{t2}'
    hedge_ratio, intercept, r_squared = estimate_hedge_ratio(
        t1, t2, train_prices
    )
    hedge_ratios[pair_name] = {
        'ticker1':       t1,
        'ticker2':       t2,
        'hedge_ratio':   round(hedge_ratio, 4),
        'intercept':     round(intercept, 4),
        'r_squared':     round(r_squared, 4)
    }
    print(f'{pair_name:<15} {hedge_ratio:>12.4f} '
          f'{intercept:>12.4f} {r_squared:>12.4f} ')

print(f'\nHedge ratios estimated for {len(hedge_ratios)} pairs.')
print('Training Period: 2014-01-01 to 2017-12-31 (209 weeks)')   

Estimating hedge ratios on training data only (2014-2017)...
These ratios will be frozen and applied to all three periods. 

Pair             Hedge Ratio    Intercept    R-Squared
----------------------------------------------------------------------
CDNS/SNPS             0.5298      -4.6507       0.9639 
KLAC/ODFL             0.2496      -0.0025       0.7223 
KLAC/SNPS             0.1060       0.2788       0.8819 
ODFL/SNPS             0.3405       5.6788       0.7856 
CDNS/KLAC             4.5418      -3.2908       0.9022 
FAST/MSFT             0.0349       6.8309       0.2852 
QCOM/TSLA            -0.3643      53.1894       0.0379 
PAYX/PEP              0.6640     -10.7111       0.8777 
ADP/PEP               1.2323     -20.0479       0.8991 
AMGN/CTAS             2.1242      67.8748       0.7181 
CDW/CTAS              1.7830       0.1472       0.9546 
AMGN/CDW              1.1213      70.3926       0.6664 

Hedge ratios estimated for 12 pairs.
Training Period: 2014-01-01 to 2017-12-

In [5]:
# Building spreads for all three periods using frozen training hedge ratios
def build_spread_period(ticker1, ticker2, hedge_ratio, intercept, period_prices):
    """
    Constructs the price spread for a given period using the 
    hedge ratio estimated on training data only.

    Spread = Price(ticker1) - hedge_ratio * Price(ticker2)

    The hedge_ratio and intercept are passed in from the frozen
    training-period estimates. They are never re-estimated here.
    """
    combined = pd.concat(
        [period_prices[ticker1], period_prices[ticker2]], axis=1
    ).dropna()
    combined.columns = [ticker1, ticker2]

    spread = combined[ticker1] - hedge_ratio * combined[ticker2]

    return spread

# Building spreads for all 12 pairs across all three periods
print('Building spreads for all 12 pairs across all three periods...')
print('Using frozen training-period hedge ratios throughout.\n')

spread_dict={}

for pair_name, params in hedge_ratios.items():
    t1          = params['ticker1']
    t2          = params['ticker2']
    hedge_ratio = params['hedge_ratio']
    intercept   = params['intercept']

    train_spread = build_spread_period(t1, t2, hedge_ratio,intercept, train_prices)
    val_spread = build_spread_period(t1, t2, hedge_ratio,intercept, val_prices)
    test_spread = build_spread_period(t1, t2, hedge_ratio,intercept, test_prices)

    spread_dict[pair_name] = {
        'train': train_spread,
        'val':   val_spread,
        'test':  test_spread,
        'ticker1': t1,
        'ticker2': t2,
        'hedge_ratio': hedge_ratio
    }

# Confirming spread lengths
print(f'{'Pair':<15} {'Train Weeks':>12} {'Val Weeks':>12} {'Test Weeks':>12}')
print("-"*55)

for pair_name, data in spread_dict.items():
    print(f"{pair_name:<15} "
          f"{len(data['train']):>12} "
          f"{len(data['val']):>12} "
          f"{len(data['test']):>12}")

print(f'\nSpreads built for {len(spread_dict)} pairs across 3 periods.')    

Building spreads for all 12 pairs across all three periods...
Using frozen training-period hedge ratios throughout.

Pair             Train Weeks    Val Weeks   Test Weeks
-------------------------------------------------------
CDNS/SNPS                209          157          156
KLAC/ODFL                209          157          156
KLAC/SNPS                209          157          156
ODFL/SNPS                209          157          156
CDNS/KLAC                209          157          156
FAST/MSFT                209          157          156
QCOM/TSLA                209          157          156
PAYX/PEP                 209          157          156
ADP/PEP                  209          157          156
AMGN/CTAS                209          157          156
CDW/CTAS                 209          157          156
AMGN/CDW                 209          157          156

Spreads built for 12 pairs across 3 periods.


In [6]:
# Regime detection using rolling spread volatility
def detect_regimes(spread, vol_window=26, vol_threshold=1.5):
    """
    Detects high and low volatility regimes in a spread series.

    A high volatility regime is defined as periods where the rolling 
    standard deviation of the spread exceeds vol_threshold times 
    its own historical median.

    In high volatility regimes the entry threshold is widened to 
    avoid being whipsawed by large erratic spread moves.
    In low volatility regimes the entry threshold is narrow to capture
    smaller but more reliable mean-reversion moves.

    Returns a Series of regime labels: 'high or low'
    """
    # Rolling std. dev. of the spread
    rolling_vol = spread.rolling(window=vol_window).std()

    # Median volatility over the available history
    median_vol = rolling_vol.median()

    # Labeling each week as 'high' or 'low' volatility regimes
    regime = pd.Series(index=spread.index, dtype=str)
    regime[rolling_vol > vol_threshold * median_vol] = 'high'
    regime[rolling_vol <= vol_threshold * median_vol] = 'low'
    regime[rolling_vol.isna()] = 'low'

    return regime, rolling_vol, median_vol

# Computing regimes for all pairs across all three periods
VOL_WINDOW = 26
VOL_THRESHOLD = 1.5

# Dynamic threshold by regime
HIGH_VOL_ENTRY = 2.5 
LOW_VOL_ENTRY = 1.5
STANDARD_ENTRY = 2.0

print('Computing volatility regimes for all 12 pairs...')
print(f'  Volatility window:     {VOL_WINDOW} weeks')
print(f'  High vol threshold:    {VOL_THRESHOLD}x median volatility')
print(f'  High vol entry:        ±{HIGH_VOL_ENTRY}σ')
print(f'  Low vol entry:         ±{LOW_VOL_ENTRY}σ')
print(f'  Standard entry:        ±{STANDARD_ENTRY}σ\n')

regime_dict = {}

for pair_name, data in spread_dict.items():
    pair_regimes = {}

    for period in ['train', 'val', 'test']:
        spread = data[period]
        regime, rolling_vol, median_vol = detect_regimes(
            spread, 
            vol_window=VOL_WINDOW,
            vol_threshold=VOL_THRESHOLD
        )

        pair_regimes[period] = {
            'regime':      regime,
            'rolling_vol': rolling_vol,
            'median_vol': float(median_vol)
        }

    regime_dict[pair_name] = pair_regimes

# Printing regime summary
print(f'{'Pair':<15} {'Train High%':>12} {'Val High%':>12} {'Test High%':>12}')
print("-"*55)

for pair_name, periods in regime_dict.items():
    summaries = []
    for period in ['train', 'val', 'test']:
        regime = regime_dict[pair_name][period]['regime']
        pct_high = (regime == 'high').sum() / len(regime) * 100
        summaries.append(f'{pct_high:>11.1f}%')
    print(f'{pair_name:<15}{summaries[0]} {summaries[1]} {summaries[2]}')

print('\nRegime detection complete')
print('High Volatility weeks will use wider entry threshold.')
print('Low Volatiltity weeks will use narrower entry threshold.')        

Computing volatility regimes for all 12 pairs...
  Volatility window:     26 weeks
  High vol threshold:    1.5x median volatility
  High vol entry:        ±2.5σ
  Low vol entry:         ±1.5σ
  Standard entry:        ±2.0σ

Pair             Train High%    Val High%   Test High%
-------------------------------------------------------
CDNS/SNPS              6.7%         8.3%         5.8%
KLAC/ODFL             27.3%        22.9%        19.9%
KLAC/SNPS              9.1%        10.2%         0.0%
ODFL/SNPS             11.5%         1.9%        19.2%
CDNS/KLAC             20.6%         8.9%         9.0%
FAST/MSFT             19.6%         8.3%         5.1%
QCOM/TSLA              3.3%        16.6%        16.7%
PAYX/PEP              23.0%        12.7%         2.6%
ADP/PEP               26.8%        15.3%        13.5%
AMGN/CTAS              9.6%        12.7%         0.0%
CDW/CTAS               7.2%        17.2%         8.3%
AMGN/CDW              14.8%        22.9%        12.2%

Regime detectio

In [7]:
# Regime-aware Z-score computation
def compute_zscore_period(spread, window=52):
    """
    Computes rolling z-score for a spread series.
    Identical to Notebook 3 but applied per period.
    """
    rolling_mean = spread.rolling(window=window).mean()
    rolling_std = spread.rolling(window=window).std()
    zscore = (spread - rolling_mean) / rolling_std
    return zscore, rolling_mean, rolling_std

def get_entry_threshold(regime, high_entry=HIGH_VOL_ENTRY,
                        low_entry = LOW_VOL_ENTRY):
    """
    Returns the appropriate entry threshold based on current regime.
    High volatility regime uses wider threshold to avoid whipsaw.
    Low volatillity regime uses narrower threshold to capture more signal.
    """
    if  regime == 'high':
        return high_entry
    else:
        return low_entry

# Walk-forward backtest with regime-aware threshold
def backtest_period(pair_name, period, spread_dict,
                    regime_dict, zscore_window=ZSCORE_WINDOW,
                    exit_threshold=EXIT_THRESHOLD,
                    stop_loss=STOP_LOSS):
    """
    Runs the pairs trading backtest for one pair over one period.
    Uses regime-aware entry threshols instead of fixed thresholds.

    Position conventions:
    +1 = Long spread
    -1 = Short spread
     0 = Flat
     """
    spread = spread_dict[pair_name][period]
    regime = regime_dict[pair_name][period]['regime']

    # Computing z-score for this period
    zscore, _, _ = compute_zscore_period(spread, window=zscore_window)

    # Aligning spread, zscore and regime on shared index
    combined = pd.concat([spread,zscore,regime], axis=1).dropna()
    combined.columns = ['spread','zscore','regime']

    # Initializing tracking variables
    position     = 0
    positions    = []
    spread_vals  = []
    pnl          = []
    prev_spread  = None
    entry_thresh_used = []

    # Walk-forward
    for i in range(len(combined)):
        row           = combined.iloc[i]
        z             = float(row['zscore'])
        s             = float(row['spread'])
        reg           = row['regime']
        entry_thresh  = get_entry_threshold(reg)
        weekly_pnl    = 0.0

        # Calculating PnL from existing position
        if position != 0 and prev_spread is not None:
            spread_change = s - prev_spread
            weekly_pnl    = position * spread_change

        # Exit Logic
        if position == 1:
            if z >= -exit_threshold or z <= -stop_loss:
                position = 0

        elif position == -1:
            if z <= exit_threshold or z >= stop_loss:
                position = 0

        # Entry Logic
        if position == 0:
            if z <= -entry_thresh:
                position = 1
            elif z >= entry_thresh:
                position = -1

        # Record state
        positions.append(position)
        spread_vals.append(s)
        pnl.append(weekly_pnl)
        entry_thresh_used.append(entry_thresh)
        prev_spread = s

    # Building results DataFrame 
    results = pd.DataFrame({
        'spread':         spread_vals,
        'zscore':         combined['zscore'].values,
        'regime':         combined['regime'].values,
        'entry_thresh':   entry_thresh_used,
        'positions':      positions,
        'weekly_pnl':     pnl
    }, index=combined.index)

    results['cumulative_pnl'] = results['weekly_pnl'].cumsum()

    return results

# Running backtests for all 12 pairs across all three periods
print('Running regime-aware backtest for all 12 pairs')
print('across all three periods...\n')

all_backtest_results = {}

for pair_name in spread_dict.keys():
    all_backtest_results[pair_name] = {}
    for period in ['train', 'val','test']:
        all_backtest_results[pair_name][period] = backtest_period(
            pair_name, period, spread_dict, regime_dict
        )

print('Backtest Complete!\n')

# Printing PnL summary across periods
print(f'{'Pair':<15} {'Train PnL':>10}{'Val PnL':>10} '
      f'{'Test PnL':>10} {'Total PnL':>10}')
print('-'*50)

for pair_name,periods in all_backtest_results.items():
    train_pnl = periods['train']['cumulative_pnl'].iloc[-1]
    val_pnl   = periods['val']['cumulative_pnl'].iloc[-1]
    test_pnl  = periods['test']['cumulative_pnl'].iloc[-1]
    total_pnl = train_pnl + val_pnl + test_pnl

    print(f'{pair_name:<15} {train_pnl:>10.2f} {val_pnl:<10.2f} '
          f'{test_pnl:>10.2f} {total_pnl:>10.2f}')

Running regime-aware backtest for all 12 pairs
across all three periods...

Backtest Complete!

Pair             Train PnL   Val PnL   Test PnL  Total PnL
--------------------------------------------------
CDNS/SNPS            -0.35 3.39            45.28      48.32
KLAC/ODFL             1.36 6.43            23.47      31.25
KLAC/SNPS             3.93 1.74            18.95      24.62
ODFL/SNPS            -0.77 26.50           75.15     100.87
CDNS/KLAC             6.97 11.29           61.85      80.10
FAST/MSFT             3.10 -1.00            9.02      11.12
QCOM/TSLA           -10.02 -109.61         19.93     -99.70
PAYX/PEP             18.65 5.58            29.69      53.92
ADP/PEP              23.43 42.05           72.34     137.82
AMGN/CTAS            61.68 37.52           60.09     159.29
CDW/CTAS             16.34 30.47           50.59      97.40
AMGN/CDW             66.02 41.80           55.41     163.23


In [10]:
# Performance Metrics
def compute_metrics_period(pair_name, period, results):
    """
    Computes performance metrics for one pair over one period.
    Identical metric definitions to Notebook 3.
    """
    weekly_pnl     = results['weekly_pnl']
    cumulative_pnl = results['cumulative_pnl']
    positions      = results['positions']

    # Trade Identification
    position_changes = positions.diff().fillna(0)
    trade_entries    = (position_changes != 0) & (positions != 0)
    n_trades         = int(trade_entries.sum())

    # Return metrics
    total_pnl         = float(cumulative_pnl.iloc[-1])
    active_pnl        = weekly_pnl[positions != 0]
    avg_weekly_pnl    = float(active_pnl.mean()) if len(active_pnl) > 0 else 0.0
    pnl_std           = float(active_pnl.std()) if len(active_pnl) > 0 else 0.0

    # Sharpe Ratio
    if pnl_std > 0:
        sharpe = (avg_weekly_pnl / pnl_std) * np.sqrt(52)
    else:
        sharpe = 0.0

    # Maximum Drawdown
    rolling_max  = cumulative_pnl.cummax()
    drawdown     = cumulative_pnl - rolling_max
    max_drawdown = float(drawdown.min())

    # Win Rate
    win_rate = float((active_pnl > 0).sum() / len(positions)) \
               if len(active_pnl) > 0 else 0.0

    # Time in market
    time_in_market = float((positions != 0).sum() / len(positions))

    # Regime usage
    if 'regime' in results.columns:
        high_vol_weeks = (results['regime'] == 'high').sum()
        pct_high_vol   = float(high_vol_weeks  / len(results) * 100)
    else:
        pct_high_vol = 0.0

    return {
        'Pair':          pair_name,
        'Period':        period,
        'Total PnL':     round(total_pnl, 2),
        'Sharpe':        round(sharpe, 4),
        'Max Drawdown':  round(max_drawdown, 2),
        'N Trades':      n_trades,
        'Win Rate':       round(win_rate, 4),
        'Time in Market': round(time_in_market, 4),
        'High Vol %':     round(pct_high_vol, 1)
    }   

# Computing metrics for all pairs across all three periods
print('Computing perormance metrics for all 12 pairs')
print('across all three periods..\n')

all_metrics = []

for pair_name, periods in all_backtest_results.items():
    for period in ['train','val','test']:
        metrics = compute_metrics_period(
            pair_name, period, periods[period]
        )
        all_metrics.append(metrics)
        
metrics_df = pd.DataFrame(all_metrics)

# Printing metrics by period
for period in ['train','val','test']:
    period_df = metrics_df[metrics_df['Period'] == period].copy()
    period_df = period_df.sort_values('Total PnL', ascending=False)
    period_df = period_df.set_index('Pair')

    print(f'\n{'='*75}')
    print(f'{period.upper()} PERIOD METRICS')
    print(f'\n{'='*75}')
    print(period_df[['Total PnL', 'Sharpe', 'Max Drawdown',
                     'N Trades', 'Win Rate','High Vol %']].to_string())

# Cross-period Sharpe Comparison
print(f'\n\n{'='*75}')
print('SHARPE RATIO ACROSS ALL THREE PERIODS')
print(f'{'='*75}')
print(f'\n{'Pair':<15} {'Train Sharpe':>13} {'Val Sharpe':>13} '
      f'{'Test Sharpe':>13} {'Avg Sharpe':>13}')
print('-'*60)

for pair_name in spread_dict.keys():
    sharpes = []
    for period in ['train', 'val', 'test']:
        s = metrics_df[
             (metrics_df['Pair'] == pair_name) &
             (metrics_df['Period'] == period)
        ]['Sharpe'].values[0]
        sharpes.append(s)
        
    avg_sharpe = np.mean(sharpes)
    tier = 'T1' if (pair_name.split('/')[0],
                    pair_name.split('/')[1]) in tier1_pairs else 'T2'

    print(f'{pair_name:<15} {sharpes[0]:>13.4f} {sharpes[1]:>13.4f} '
          f'{sharpes[2]:>13.4f} {avg_sharpe:>13.4f} [{tier}]')

# Saving metrics
metrics_df.to_csv('three_period_metrics.csv', index=False)
print('\nMetrics saved to three_period_metrics.csv')
      

Computing perormance metrics for all 12 pairs
across all three periods..


TRAIN PERIOD METRICS

           Total PnL  Sharpe  Max Drawdown  N Trades  Win Rate  High Vol %
Pair                                                                      
AMGN/CDW     66.0200  2.2058      -14.6400         6    0.1835     13.9000
AMGN/CTAS    61.6800  1.8569      -14.3700         6    0.1456      8.2000
ADP/PEP      23.4300  1.0597       -8.1700         6    0.1772     35.4000
PAYX/PEP     18.6500  0.6861       -2.4700         7    0.1899     30.4000
CDW/CTAS     16.3400  0.4271       -5.6300         7    0.1899      9.5000
CDNS/KLAC     6.9700 -0.0987       -5.8900         4    0.2025     27.2000
KLAC/SNPS     3.9300  0.6845       -1.2200         5    0.1899     12.0000
FAST/MSFT     3.1000  0.6314       -0.9100         6    0.1772     25.9000
KLAC/ODFL     1.3600 -0.0214       -1.1800         6    0.2405     36.1000
CDNS/SNPS    -0.3500 -0.4977       -3.7200         4    0.3165      7.0000
ODF